In [1]:
import os
import re
import csv
from collections import defaultdict
from statistics import mean, stdev

def parse_log_file_last_metrics(path, dataset):
    eval_metrics = {}
    with open(path, encoding="utf-8") as f:
        lines = f.readlines()
    # 倒序查找最后一个 start to eval
    i = len(lines) - 1
    while i >= 0:
        if lines[i].strip().startswith("start to eval"):
            # 期待接下来三行分别是 hit20、hit50、hit100
            if i + 1 < len(lines) and lines[i+1].startswith("hit20"):
                m = re.search(r"hit20\s+([0-9.eE+-]+)", lines[i+1])
                if m: eval_metrics["hit20"] = float(m.group(1))
            if i + 2 < len(lines) and lines[i+2].startswith("hit50"):
                m = re.search(r"hit50\s+([0-9.eE+-]+)", lines[i+2])
                if m: eval_metrics["hit50"] = float(m.group(1))
            if i + 3 < len(lines) and lines[i+3].startswith("hit100"):
                m = re.search(r"hit100\s+([0-9.eE+-]+)", lines[i+3])
                if m: eval_metrics["hit100"] = float(m.group(1))
            if i + 4 < len(lines) and lines[i+4].startswith("roc_auc"):
                if 'mrr_pess' in lines[i+4]:
                    fields = ['roc_auc', 'pr_auc', 'f1', 'mrr_pess', 'mrr_opt']
                else:
                    fields = ['roc_auc', 'pr_auc', 'f1', 'mrr']
                nums = [float(x) for x in re.findall(r'(?<![A-Za-z])[+-]?(?:\d+\.\d*|\.\d+|\d+)(?:[eE][+-]?\d+)?(?![A-Za-z])', lines[i+4])]
                if 'mrr_pess' in lines[i+4]:
                    nums = nums[-5:]
                else:
                    nums = nums[-4:]
                result_dict = dict(zip(fields, nums))

                print(result_dict)
                
                for k, v in result_dict.items():
                    eval_metrics[k] = v

            if eval_metrics:
                break
        i -= 1
    return eval_metrics

# ==== 用户指定参数 ====
datasets = ["ogbl_citation2"] # "amazon", "ogbl_citation2", "cora", "pubmed", "citeseer", "cora_ml", "icews18_max", ]
ratios = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
seeds = [1] # 之后记得改回 [1, 2, 3, 4, 5]
log_dir = "."

# ==== 处理每个数据集的日志文件 ====
for dataset in datasets:
    metric_bucket = defaultdict(list)
    metric_keys = set()
    
    for ratio in ratios:
        for seed in seeds:
            log_file = f"s{seed}-r{ratio}.log"
            log_path = os.path.join(log_dir, dataset, log_file)
            if not os.path.isfile(log_path):
                print(f"[WARN] 缺失: {log_path}")
                continue
            metrics = parse_log_file_last_metrics(log_path, dataset)
            if not metrics:
                print(f"[WARN] 无 test 指标: {log_path}")
                continue
            metric_bucket[ratio].append(metrics)
            metric_keys.update(metrics.keys())

    # ==== 结果输出 ====
    csv_path = f"./results/{dataset}_result.csv"
    metric_keys = sorted(metric_keys)
    header = ["split_ratio"] + [f"{k}" for k in metric_keys]

    with open(csv_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=header)
        writer.writeheader()
        for ratio in sorted(metric_bucket):
            row = {"split_ratio": ratio}
            for k in metric_keys:
                vals = [m[k] for m in metric_bucket[ratio] if k in m]
                if vals:
                    mu = round(round(mean(vals), 6) * 100, 4)
                    sd = round(round(stdev(vals), 6) * 100 if len(vals) > 1 else 0.0, 4)
                    row[k] = f"{mu} ± {sd}"
                else:
                    row[k] = ""
            writer.writerow(row)
    print(f"[INFO] 写入完毕: {csv_path}")

{'roc_auc': 0.844653543228556, 'pr_auc': 0.6901349717934973, 'f1': 0.6096434798402296, 'mrr_pess': 0.22625933878405508, 'mrr_opt': 0.22648238968384785}
{'roc_auc': 0.8951710018729923, 'pr_auc': 0.8019015958278137, 'f1': 0.7136454919191018, 'mrr_pess': 0.40000361113559096, 'mrr_opt': 0.40017733283755175}
{'roc_auc': 0.9335099278078142, 'pr_auc': 0.8754270793054053, 'f1': 0.7980216774477517, 'mrr_pess': 0.5287088109588506, 'mrr_opt': 0.5288995119233063}
{'roc_auc': 0.9497373766156597, 'pr_auc': 0.9051761770389272, 'f1': 0.8355233614197027, 'mrr_pess': 0.5809304563648785, 'mrr_opt': 0.5811044037605748}
{'roc_auc': 0.9562688396480851, 'pr_auc': 0.9168285645157435, 'f1': 0.8505687620791927, 'mrr_pess': 0.6093467409634301, 'mrr_opt': 0.6095155439841943}
{'roc_auc': 0.9674017725633892, 'pr_auc': 0.9345025537070251, 'f1': 0.8729495046230248, 'mrr_pess': 0.6377066010575417, 'mrr_opt': 0.6378725365173232}
{'roc_auc': 0.9743517753989108, 'pr_auc': 0.9455427915332899, 'f1': 0.8873461948920867, 'mr